# Array programming exercises

**Name:**

**Collaborators:**

In [1]:
import numpy as np

## Exercise 1: The Empirical cdf

Given a collection of real-valued data $y_1,\ldots,y_n$, the empirical cumulative distribution function (ecdf) is 
$$
F_n(x) = \frac{1}{n}\sum_{i=1}^n \mathbb{1}\{y_i\le x\}.
$$
We wish to implement a function `make_ecdf` which takes in `data` = $(y_1,\ldots,y_n)$ and returns a vectorized function `Fn` which can take in either a single $x$ or an array of $x$s.

The naïve implementation of `make_ecdf` directly computes this sum for each value of $x$. This is already implemented below. **Your goal in this exercise is to implement a faster version `make_ecdf_fast`**, which sorts the `data` and then performs a binary search to compute $\sum_{i=1}^n \mathbb{1}\{y_i\le x\}$ for each $x$. You should do this using `np.searchsorted`.

In [2]:
?np.searchsorted

Signature:       np.searchsorted(a, v, side='left', sorter=None)
Call signature:  np.searchsorted(*args, **kwargs)
Type:            _ArrayFunctionDispatcher
String form:     <function searchsorted at 0x120cd8900>
File:            ~/Projects/PhD_Courses/stats607-studios/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py
Docstring:      
Find indices where elements should be inserted to maintain order.

Find the indices into a sorted array `a` such that, if the
corresponding elements in `v` were inserted before the indices, the
order of `a` would be preserved.

Assuming that `a` is sorted:

======  ============================
`side`  returned index `i` satisfies
======  ============================
left    ``a[i-1] < v <= a[i]``
right   ``a[i-1] <= v < a[i]``
======  ============================

Parameters
----------
a : 1-D array_like
    Input array. If `sorter` is None, then it must be sorted in
    ascending order, otherwise `sorter` must be an array of indices
    that 

In [3]:
def make_ecdf(data):
    """
    Create an empirical CDF function from data.
    
    Parameters:
    -----------
    data : array-like
        The input data points
    
    Returns:
    --------
    Fn : function
        A vectorized function that evaluates the ECDF at any point(s) x
        Returns the proportion of data points <= x
    
    Example:
    --------
    >>> data = [1, 2, 2, 3, 4, 4, 4, 5]
    >>> Fn = make_ecdf(data)
    >>> Fn(3)
    0.5
    >>> Fn([1, 2.5, 4])
    array([0.125, 0.375, 0.875])
    """
    data_array = np.asarray(data)
    n = len(data_array)
    
    def Fn(x):
        x = np.asarray(x)
        return np.sum(data_array <= x[..., np.newaxis], axis=-1) / n
    
    return Fn

def make_ecdf_fast(data, sorted=False):
    """
    Create an empirical CDF function from data.
    
    Parameters:
    -----------
    data : array-like
        The input data points
    sorted : bool, optional (default=False)
        If True, assumes data is already sorted and skips sorting step
    
    Returns:
    --------
    Fn : function
        A vectorized function that evaluates the ECDF at any point(s) x
        Returns the proportion of data points <= x
    
    Example:
    --------
    >>> data = [1, 2, 2, 3, 4, 4, 4, 5]
    >>> Fn = make_ecdf(data)
    >>> Fn(3)
    0.5
    >>> Fn([1, 2.5, 4])
    array([0.125, 0.375, 0.875])
    
    >>> # If data is already sorted, skip the sorting step
    >>> sorted_data = np.sort(data)
    >>> Fn = make_ecdf(sorted_data, sorted=True)
    """

    data = np.sort(np.asarray(data)) if not sorted else np.asarray(data)
    n = len(data)
        
    def __cdf(x):
        x = np.asarray(x)
        indices = np.searchsorted(data, x, side='right')
        return indices/n
    return __cdf

cdf = make_ecdf_fast([1, 2, 2, 3, 4, 4, 4, 5])   

# 7/8

#### Testing

In [4]:
# Test that we get the right answer on a small instance
data = [1, 2, 2, 3, 4, 4, 4, 5]

Fn = make_ecdf(data)
assert Fn(3) == 0.5

Fn_fast = make_ecdf_fast(data)
assert Fn_fast(.5) == 0.
assert Fn_fast(10) == 1.0


# Test that methods give the same answer on a large instance
n, N = 100, 1000
data = np.random.rand(n)
x    = np.random.rand(N)

Fn_fast = make_ecdf_fast(data)
Fn = make_ecdf(data)

assert np.allclose(Fn(x), Fn_fast(x)), "not all close"
# x[~np.isclose(Fn(x), Fn_fast(x))], Fn(x)[~np.isclose(Fn(x), Fn_fast(x))], Fn_fast(x)[~np.isclose(Fn(x), Fn_fast(x))]

#### Profiling

In [5]:
for n in [16, 256, 65536]:
    print(f"\nn={n}")
    N = 1000
    data = np.random.rand(n)
    x    = np.random.rand(N)
    
    Fn = make_ecdf(data)
    Fn_fast = make_ecdf_fast(data)

    %timeit Fn(x)
    %timeit Fn_fast(x)


n=16
20.2 μs ± 1.02 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
5.53 μs ± 67.9 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)

n=256
149 μs ± 1.59 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
9.06 μs ± 113 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)

n=65536
24.2 ms ± 394 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
16.7 μs ± 403 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


#### (!) Question

Which implementation scales more efficiently with the sample size $n$ (i.e., the length of `data`)? Describe the computational complexity of evaluating the ecdf $F_n(x)$ for each method as a function of $n$.

**Answer:**

Definitely the ECDF fast one. 

## Exercise 2: BH Procedure in Multiple Testing

In [6]:
def BH(pvals, q=0.05, sorted=False):
    """
    Run the BH procedure at level q
    
    pvals : ndarray of shape (m, S), sorted along axis=0
    q : float or ndarray of shape (,S)
    sorted : bool, optional (default=False)
        If True, assumes pvals is already sorted and skips sorting step

    Returns # of rejections R and the p-value corresponding 
    to the last rejection tau (both ndarrays of shape (S,)). 
    """
    pvals = np.asarray(pvals)
    pvals = pvals if sorted else np.sort(pvals, axis=0)

    m, S = pvals.shape
    thresh = q * np.arange(1, m+1) / m
    results = np.zeros((S, 2))
    for s in range(S):
        if np.any(pvals[:, s] <= thresh):
            results[s, 0] = np.max(np.where(pvals[:, s] <= thresh)[0]) + 1 
            results[s, 1] = pvals[int(results[s, 0])-1, s]
        else:
            results[s, :] = 0
    return results

def BH_fast(pvals, q=0.05, sorted=False):
    """
    Run the BH procedure at level q
    
    pvals : ndarray of shape (m, S), sorted along axis=0
    q : float or ndarray of shape (,S)
    sorted : bool, optional (default=False)
        If True, assumes pvals is already sorted and skips sorting step

    Returns # of rejections R and the p-value corresponding 
    to the last rejection tau (both ndarrays of shape (S,)). 
    """
    pvals = np.sort(pvals, axis=0) if not sorted else pvals
    m, _ = pvals.shape
    ranks = np.arange(1, m+1)
    q_adj = ranks/m * q

    passes = pvals <= q_adj[:, np.newaxis]
    pass_count = np.sum(passes, axis=0)
    highest_p = np.max(pvals * passes, axis=0)
    return np.stack((pass_count, highest_p), axis=1)
    

In [7]:
p = np.random.beta(.1, 1, size=(10, 100))
np.allclose(BH(p), BH_fast(p))

True

In [ ]:
timing_slow  = %timeit -o BH(p)
timing_fast  = %timeit -o BH_fast(p)
print(f"{round(timing_slow.average / timing_fast.average)} times faster!")

275 μs ± 3.34 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
